# Study Guide: Autoregressive HMM in Dynamax

This is an annotated walkthrough of the `autoregressive_hmm.ipynb` notebook.
Each section explains **what** the code does and **why**, with an eye toward
extending this to ECoG data with multiple time lags.

---

## 1. The Model

An **Autoregressive HMM (AR-HMM)** combines two ideas:

1. **Hidden Markov Model**: A discrete latent state $z_t \in \{1, \ldots, K\}$ evolves as a Markov chain with transition matrix $P$.
2. **Autoregressive emissions**: The observation $y_t$ depends on the *previous* observation(s), with the dynamics governed by the current discrete state.

For **lag-1**, the model is:

$$z_t \mid z_{t-1} \sim \text{Categorical}(P_{z_{t-1}})$$
$$y_t \mid y_{t-1}, z_t \sim \mathcal{N}(A_{z_t} y_{t-1} + b_{z_t}, Q_{z_t})$$

For **lag-L** (general), it becomes:

$$y_t \mid y_{t-1}, \ldots, y_{t-L}, z_t \sim \mathcal{N}\left(\sum_{\ell=1}^{L} W_{z_t, \ell}\, y_{t-\ell} + b_{z_t},\; Q_{z_t}\right)$$

### Why this matters for ECoG

ECoG signals have **temporal autocorrelation** spanning multiple timesteps.
A lag-1 model assumes the signal at time $t$ only depends on time $t-1$,
but neural signals often have dependencies reaching further back.
Using `num_lags > 1` lets the model capture these longer-range dependencies
within each regime.

---
## 2. Setup and Imports

In [ ]:
%%capture
try:
    import dynamax
except ModuleNotFoundError:
    print('installing dynamax')
    %pip install -q dynamax[notebooks]
    import dynamax

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import seaborn as sns

from dynamax.hidden_markov_model import LinearAutoregressiveHMM
from dynamax.utils.plotting import gradient_cmap
from dynamax.utils.utils import random_rotation

In [ ]:
# Plotting setup
sns.set_style("white")

color_names = [
    "windows blue", "red", "amber", "faded green", "dusty purple",
    "orange", "brown", "pink"
]
colors = sns.xkcd_palette(color_names)
cmap = gradient_cmap(colors)

---
## 3. Building the Transition Matrix

The transition matrix $P$ controls how the discrete state $z_t$ evolves.
Here we build a matrix where each state has:
- **High self-transition probability** (stays in the same state)
- **Small probability** of advancing to the next state (cyclically)

This creates a sequential switching pattern: state 0 → 1 → 2 → 3 → 4 → 0 → ...

### Key insight
The `transition_probs = (jnp.arange(K)**10)` trick creates a sharply peaked distribution.
After normalization, most of the mass is on the largest index (self-transition),
with a tiny amount on the next state.

In [ ]:
num_states = 5

# Create transition probabilities: [0^10, 1^10, 2^10, 3^10, 4^10] -> normalized
# This makes self-transition probability very high (~0.96)
transition_probs = (jnp.arange(num_states)**10).astype(float)
transition_probs /= transition_probs.sum()

print("Transition probabilities (before rolling):")
print(transition_probs)
print(f"\nSelf-transition probability: {transition_probs[-1]:.4f}")
print(f"Next-state probability: {transition_probs[-2]:.4f}")

In [ ]:
# Build the full transition matrix by rolling the probabilities for each row
# Each row k has the same distribution but shifted so that state k has the
# highest probability (self-transition) and state (k+1) % K is next most likely
transition_matrix = jnp.zeros((num_states, num_states))
for k, p in enumerate(transition_probs[::-1]):
    transition_matrix += jnp.roll(p * jnp.eye(num_states), k, axis=1)

plt.figure(figsize=(4, 3))
plt.imshow(transition_matrix, vmin=0, vmax=1, cmap="Greys")
plt.xlabel("next state")
plt.ylabel("current state")
plt.title("Transition Matrix P")
plt.colorbar()
plt.tight_layout()

# Verify: each row sums to 1
print("Row sums:", transition_matrix.sum(axis=1))
print("\nFull matrix:")
print(jnp.round(transition_matrix, 4))

---
## 4. Constructing the Emission Parameters (Dynamics)

This is the **core of the AR-HMM**. Each discrete state $k$ defines a different
linear dynamical system:

$$y_t = A_k y_{t-1} + b_k + \epsilon_t, \quad \epsilon_t \sim \mathcal{N}(0, Q_k)$$

The parameters for each state $k$ are:
- **$A_k$** (`weights`): dynamics matrix — how the previous observation maps to the current one
- **$b_k$** (`biases`): offset/fixed point attractor
- **$Q_k$** (`covariances`): noise covariance

### What `random_rotation` does
It creates a rotation matrix with angle `theta`. Multiplying by 0.8 makes
eigenvalues have magnitude 0.8 < 1, so the system is **stable** (converges
to a fixed point rather than diverging).

### Fixed points
Each state's dynamics spiral toward a **fixed point** $y^* = (I - A_k)^{-1} b_k$.
The biases are set to points on the unit circle, so the fixed points form a ring.

In [ ]:
emission_dim = 2  # 2D observations (easy to visualize)
num_lags = 1      # Start with lag-1

# Create state-dependent dynamics parameters
keys = jr.split(jr.PRNGKey(0), num_states)
angles = jnp.linspace(0, 2 * jnp.pi, num_states, endpoint=False)

theta = jnp.pi / 25  # Rotation angle per timestep

# A_k = 0.8 * R(theta) — a scaled rotation matrix
# The 0.8 factor ensures |eigenvalues| < 1 (stability)
weights = jnp.array([0.8 * random_rotation(key, emission_dim, theta=theta) for key in keys])

# b_k = [cos(angle_k), sin(angle_k)] — points on unit circle
biases = jnp.column_stack([jnp.cos(angles), jnp.sin(angles), jnp.zeros((num_states, emission_dim - 2))])

# Q_k = 0.001 * I — very small noise
covariances = jnp.tile(0.001 * jnp.eye(emission_dim), (num_states, 1, 1))

# Fixed point: y* = (I - A_k)^{-1} b_k
stationary_points = jax.vmap(jnp.linalg.solve)(jnp.eye(emission_dim) - weights, biases)

print("=== Parameter shapes ===")
print(f"weights (A_k):      {weights.shape}  — (num_states, emission_dim, emission_dim * num_lags)")
print(f"biases (b_k):       {biases.shape}  — (num_states, emission_dim)")
print(f"covariances (Q_k):  {covariances.shape}  — (num_states, emission_dim, emission_dim)")
print(f"\n=== Fixed points ===")
for k in range(num_states):
    print(f"State {k}: y* = [{stationary_points[k, 0]:.2f}, {stationary_points[k, 1]:.2f}]")

In [ ]:
# Let's inspect one dynamics matrix to understand it
print("A_0 (dynamics matrix for state 0):")
print(weights[0])
eigenvalues = jnp.linalg.eigvals(weights[0])
print(f"\nEigenvalues: {eigenvalues}")
print(f"Magnitudes: {jnp.abs(eigenvalues)}")
print(f"(All < 1 → stable, spiraling inward)")

### Visualizing the dynamics as flow fields

For 2D data, we can plot $f_k(y) - y = (A_k - I)y + b_k$ as a vector field.
The arrows show the direction the system would push from each point.
Notice they all spiral inward toward the fixed point.

In [ ]:
if emission_dim == 2:
    lim = 5
    x = jnp.linspace(-lim, lim, 10)
    y = jnp.linspace(-lim, lim, 10)
    X, Y = jnp.meshgrid(x, y)
    xy = jnp.column_stack((X.ravel(), Y.ravel()))

    fig, axs = plt.subplots(1, num_states, figsize=(2 * num_states, 4))
    for k in range(num_states):
        A, b = weights[k], biases[k]
        # Flow: f(y) - y = (A-I)y + b
        dxydt_m = xy.dot(A.T) + b - xy
        axs[k].quiver(xy[:, 0], xy[:, 1], dxydt_m[:, 0], dxydt_m[:, 1],
                       color=colors[k % len(colors)])
        axs[k].plot(stationary_points[k, 0], stationary_points[k, 1], 'k*', markersize=10)
        axs[k].set_xlabel('$y_1$')
        axs[k].set_xticks([])
        if k == 0:
            axs[k].set_ylabel("$y_2$")
        axs[k].set_yticks([])
        axs[k].set_aspect("equal")
        axs[k].set_title(f"State {k}")
    plt.suptitle("Flow fields for each state (★ = fixed point)", y=1.02)
    plt.tight_layout()

---
## 5. Creating the Model and Sampling

Now we assemble the model and sample from it. Key steps:

1. **Create** a `LinearAutoregressiveHMM` object (defines the model structure)
2. **Initialize** with our hand-crafted parameters
3. **Sample** a trajectory of states and emissions
4. **Compute inputs** — this is the crucial step that creates the lagged features

### How `compute_inputs` works (this is key for multi-lag!)

The AR-HMM is implemented as a special case of a **linear regression HMM**.
The trick: treat lagged observations as "inputs" to a regression:

$$y_t = W_{z_t} \underbrace{[y_{t-1}; y_{t-2}; \ldots; y_{t-L}]}_{\text{inputs}} + b_{z_t} + \epsilon_t$$

For lag-1 with 2D data: inputs are shape `(T, 2)` — just the previous observation.
For lag-2 with 2D data: inputs are shape `(T, 4)` — concatenation of $[y_{t-1}, y_{t-2}]$.

The source code (`arhmm.py:234-254`):
```python
def compute_inputs(self, emissions, prev_emissions=None):
    padded_emissions = jnp.vstack((prev_emissions, emissions))  # prepend zeros
    return jnp.column_stack([
        padded_emissions[lag:lag+num_timesteps]
        for lag in reversed(range(self.num_lags))
    ])
```
This creates a sliding window of past observations, zero-padded at the start.

In [ ]:
# Step 1: Create the model object
true_arhmm = LinearAutoregressiveHMM(num_states, emission_dim, num_lags=num_lags)

# Step 2: Initialize with our hand-crafted parameters
true_params, _ = true_arhmm.initialize(
    initial_probs=jnp.ones(num_states) / num_states,  # Uniform initial distribution
    transition_matrix=transition_matrix,
    emission_weights=weights,
    emission_biases=biases,
    emission_covariances=covariances
)

# Step 3: Sample!
time_bins = 10000
true_states, emissions = true_arhmm.sample(true_params, jr.PRNGKey(0), time_bins)

# Step 4: Compute lagged inputs
inputs = true_arhmm.compute_inputs(emissions)

print(f"emissions shape:  {emissions.shape}  — (T, N)")
print(f"inputs shape:     {inputs.shape}  — (T, N * num_lags)")
print(f"true_states shape: {true_states.shape}  — (T,)")
print(f"\nSanity check: inputs[t] should equal emissions[t-1]")
print(f"  inputs[5]     = {inputs[5]}")
print(f"  emissions[4]  = {emissions[4]}")
print(f"  Match: {jnp.allclose(inputs[5], emissions[4])}")

### How sampling works internally

Looking at the source (`arhmm.py:186-232`), sampling uses `jax.lax.scan` to iterate:

```
carry = (prev_state, prev_emissions)   # prev_emissions shape: (num_lags, emission_dim)
for t = 1, ..., T:
    state ~ Categorical(P[prev_state])
    emission ~ N(W[state] @ flatten(prev_emissions) + b[state], Q[state])
    prev_emissions = [emission, prev_emissions[:-1]]   # slide the window
```

The `prev_emissions` buffer is a FIFO queue of the last `num_lags` emissions.
For lag-1, it's just `(1, 2)` — a single previous observation.
For lag-3, it would be `(3, 2)` — three previous observations stacked.

In [ ]:
# Plot the sampled trajectory, color-coded by discrete state
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: 2D phase plot
for k in range(num_states):
    axes[0].plot(*emissions[true_states == k].T, 'o', color=colors[k],
                 alpha=0.75, markersize=3, label=f"state {k}")
axes[0].plot(*emissions[:1000].T, '-k', lw=0.5, alpha=0.2)
axes[0].set_xlabel("$y_1$")
axes[0].set_ylabel("$y_2$")
axes[0].set_title("Phase plot of sampled AR-HMM trajectory")
axes[0].legend(markerscale=3, fontsize=8)

# Right: time series of first 500 steps
plot_slice = slice(0, 500)
for d in range(emission_dim):
    axes[1].plot(emissions[plot_slice, d], label=f"$y_{d+1}$", alpha=0.8)
# Color the background by state
for t in range(500):
    axes[1].axvspan(t, t+1, alpha=0.15, color=colors[true_states[t]])
axes[1].set_xlabel("time")
axes[1].set_ylabel("observation")
axes[1].set_title("Time series (background = discrete state)")
axes[1].legend()

plt.tight_layout()

---
## 6. Fitting with EM (Expectation-Maximization)

Now we pretend we only observe `emissions` and try to recover the parameters.

### The EM algorithm for AR-HMMs

**E-step** (forward-backward): Compute the posterior distribution over discrete
states $p(z_t = k \mid y_{1:T})$ for each time step. This uses the standard
HMM forward-backward algorithm, treating the lagged emissions as known inputs.

**M-step** (linear regression): For each state $k$, solve a weighted least-squares
problem to find the best $W_k, b_k, Q_k$. The weights are the posterior
state probabilities from the E-step.

From the source (`linreg_hmm.py:131-186`), the sufficient statistics are:
```
sum_w[k]   = Σ_t p(z_t=k | y)                    # effective count
sum_xxT[k] = Σ_t p(z_t=k | y) * x_t x_t^T        # input outer product
sum_xyT[k] = Σ_t p(z_t=k | y) * x_t y_t^T        # input-output cross product
sum_yyT[k] = Σ_t p(z_t=k | y) * y_t y_t^T        # output outer product
```
Then the M-step solves `[W_k; b_k] = (Σ x̃x̃^T)^{-1} (Σ x̃y^T)` where `x̃ = [x; 1]`.

### Why this matters
The M-step is just linear regression — it works for **any input dimension**.
So when we switch from lag-1 (`input_dim = 2`) to lag-3 (`input_dim = 6`),
the M-step automatically handles the larger matrices. No code changes needed.

In [ ]:
# Create a new model for fitting (doesn't know the true parameters)
arhmm = LinearAutoregressiveHMM(num_states, emission_dim, num_lags=num_lags)

# Initialize with K-Means (clusters emissions to get initial biases and state assignments)
params, props = arhmm.initialize(key=jr.PRNGKey(1), method="kmeans", emissions=emissions)

print("Initial parameter shapes:")
print(f"  emission weights: {params.emissions.weights.shape}")
print(f"  emission biases:  {params.emissions.biases.shape}")
print(f"  emission covs:    {params.emissions.covs.shape}")

In [ ]:
# Run EM!
# Note: we pass `inputs` (the lagged emissions) explicitly.
# This is how the AR structure enters the standard HMM fitting machinery.
fitted_params, lps = arhmm.fit_em(params, props, emissions, inputs=inputs)

In [ ]:
# Plot convergence
true_lp = true_arhmm.marginal_log_prob(true_params, emissions, inputs=inputs)

plt.figure(figsize=(6, 4))
plt.plot(lps, label="EM")
plt.axhline(true_lp, color='k', linestyle=':', label="True model")
plt.xlabel("EM Iteration")
plt.ylabel("Log Probability")
plt.legend(loc="lower right")
plt.title("EM converges to (near) the true log-likelihood")
plt.tight_layout()

---
## 7. Comparing True vs. Inferred Parameters

In [ ]:
# Compare flow fields: true vs fitted
if emission_dim == 2:
    lim = abs(emissions).max()
    x = jnp.linspace(-lim, lim, 10)
    y = jnp.linspace(-lim, lim, 10)
    X, Y = jnp.meshgrid(x, y)
    xy = jnp.column_stack((X.ravel(), Y.ravel()))

    fig, axs = plt.subplots(2, num_states, figsize=(2.5 * num_states, 5))
    for i, (label, p) in enumerate([("True", true_params), ("Fitted", fitted_params)]):
        for j in range(num_states):
            A = p.emissions.weights[j]
            b = p.emissions.biases[j]
            dxydt_m = xy.dot(A.T) + b - xy
            axs[i, j].quiver(xy[:, 0], xy[:, 1], dxydt_m[:, 0], dxydt_m[:, 1],
                             color=colors[j % len(colors)])
            axs[i, j].set_xticks([])
            axs[i, j].set_yticks([])
            axs[i, j].set_aspect("equal")
            if i == 0:
                axs[i, j].set_title(f"State {j}", fontsize=10)
            if j == 0:
                axs[i, j].set_ylabel(label, fontsize=12, fontweight='bold')
    plt.suptitle("True vs. Fitted dynamics (may have label permutation)", y=1.02)
    plt.tight_layout()

---
## 8. Posterior State Inference

The `smoother` method runs forward-backward to compute $p(z_t = k \mid y_{1:T})$
for all $t$ and $k$. This gives us a **soft** assignment of each time step to
a discrete state.

In [ ]:
# Compute posterior
posterior = arhmm.smoother(fitted_params, emissions, inputs=inputs)

print(f"Smoothed probabilities shape: {posterior.smoothed_probs.shape}")
print(f"  = (num_timesteps - num_lags, num_states)")
print(f"  Note: first {num_lags} timestep(s) are consumed as initial lag context")

In [ ]:
# Plot true vs inferred states
plot_slice = (0, 1000)
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True)

# True discrete states
axes[0].imshow(true_states[None, num_lags:], aspect="auto", interpolation="none",
               cmap=cmap, vmin=0, vmax=len(colors)-1)
axes[0].set_xlim(plot_slice)
axes[0].set_ylabel("$z_{true}$")
axes[0].set_yticks([])
axes[0].set_title("True states vs. inferred posterior probabilities")

# Posterior probabilities
axes[1].imshow(posterior.smoothed_probs.T, aspect="auto", interpolation="none",
               cmap="Greys", vmin=0, vmax=1)
axes[1].set_xlim(plot_slice)
axes[1].set_ylabel("$p(z_t=k|y)$")
axes[1].set_yticks(range(num_states))

# Most likely state
inferred_states = jnp.argmax(posterior.smoothed_probs, axis=1)
axes[2].imshow(inferred_states[None, :], aspect="auto", interpolation="none",
               cmap=cmap, vmin=0, vmax=len(colors)-1)
axes[2].set_xlim(plot_slice)
axes[2].set_ylabel("$z_{inferred}$")
axes[2].set_yticks([])
axes[2].set_xlabel("time")

plt.tight_layout()

---
## 9. Generative Validation

Sample from the fitted model and compare to the real data visually.
If the model is good, the generated data should look qualitatively similar.

In [ ]:
sampled_states, sampled_emissions = arhmm.sample(fitted_params, jr.PRNGKey(0), time_bins)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (title, emis, states) in zip(axes, [
    ("Real data", emissions, true_states),
    ("Generated from fitted model", sampled_emissions, sampled_states)
]):
    for k in range(num_states):
        ax.plot(*emis[states == k].T, 'o', color=colors[k % len(colors)],
                alpha=0.75, markersize=3)
    ax.plot(*emis[:1000].T, '-k', lw=0.5, alpha=0.2)
    ax.set_xlabel("$y_1$")
    ax.set_ylabel("$y_2$")
    ax.set_aspect("equal")
    ax.set_title(title)
plt.tight_layout()

---
---

# Part 2: Extending to Multiple Lags

Now that we understand the lag-1 case, let's see what changes with `num_lags > 1`.

**Spoiler: almost nothing in the code changes.** The key differences are:

| | Lag-1 | Lag-L |
|---|---|---|
| Model | $y_t = A_k y_{t-1} + b_k + \epsilon$ | $y_t = \sum_{\ell=1}^L W_{k,\ell} y_{t-\ell} + b_k + \epsilon$ |
| Weight shape | `(K, N, N)` | `(K, N, N*L)` |
| Input shape | `(T, N)` | `(T, N*L)` |
| `compute_inputs` output | $[y_{t-1}]$ | $[y_{t-1}; y_{t-2}; \ldots; y_{t-L}]$ |

### Why multiple lags matter for ECoG

ECoG is typically sampled at 500-2000 Hz. Neural dynamics (e.g., beta oscillations
at ~20 Hz) span many samples. A lag-1 model at 1000 Hz only captures 1 ms of history,
but a lag-5 model captures 5 ms. For slower dynamics, you might need even more lags
(or downsample the data first).

In [ ]:
# ============================================================
# Experiment: Fit AR-HMMs with different lag orders on the
# SAME data (which was generated with lag-1).
# ============================================================

lag_values = [1, 2, 3]
results = {}

for n_lags in lag_values:
    print(f"\n{'='*50}")
    print(f"Fitting with num_lags = {n_lags}")
    print(f"{'='*50}")
    
    # Create model with this lag order
    model = LinearAutoregressiveHMM(num_states, emission_dim, num_lags=n_lags)
    
    # Compute the lagged inputs for this lag order
    lag_inputs = model.compute_inputs(emissions)
    print(f"  Input shape: {lag_inputs.shape}  (T x {emission_dim * n_lags})")
    print(f"  Weight shape will be: ({num_states}, {emission_dim}, {emission_dim * n_lags})")
    
    # Initialize and fit
    p, pr = model.initialize(key=jr.PRNGKey(1), method="kmeans", emissions=emissions)
    print(f"  Actual weight shape: {p.emissions.weights.shape}")
    
    fitted_p, log_probs = model.fit_em(p, pr, emissions, inputs=lag_inputs)
    
    results[n_lags] = {
        'model': model,
        'params': fitted_p,
        'log_probs': log_probs,
        'inputs': lag_inputs,
        'final_lp': log_probs[-1]
    }
    print(f"  Final log-prob: {log_probs[-1]:.2f}")

In [ ]:
# Compare convergence across lag orders
plt.figure(figsize=(8, 5))
for n_lags, res in results.items():
    plt.plot(res['log_probs'], label=f"num_lags={n_lags} (final: {res['final_lp']:.1f})")
plt.axhline(true_lp, color='k', linestyle=':', label=f"True model (lag-1): {true_lp:.1f}")
plt.xlabel("EM Iteration")
plt.ylabel("Log Probability")
plt.legend()
plt.title("EM convergence for different lag orders\n(data was generated with lag-1)")
plt.tight_layout()

### Interpreting the results

- **Lag-1** should recover the true model well (it matches the data-generating process)
- **Lag-2, 3** will have **slightly higher** log-likelihood on training data (more parameters = better fit)
  but the extra lag coefficients should be near zero since the true model is lag-1
- In practice with real ECoG data, you'd use **cross-validation** or **information criteria**
  (AIC/BIC) to choose the right lag order

Let's verify that the extra lag weights are indeed small:

In [ ]:
# Inspect the learned weights for lag-2 model
if 2 in results:
    W = results[2]['params'].emissions.weights
    print("Lag-2 model: learned weight matrices for state 0")
    print(f"\nFull W_0 shape: {W[0].shape} = (emission_dim, emission_dim * 2)")
    print(f"\nW_0[:, :2] (lag-1 coefficients):")
    print(W[0][:, :emission_dim])
    print(f"\nW_0[:, 2:] (lag-2 coefficients — should be ~0):")
    print(W[0][:, emission_dim:])
    
    print(f"\n--- Frobenius norms ---")
    for k in range(num_states):
        lag1_norm = jnp.linalg.norm(W[k][:, :emission_dim])
        lag2_norm = jnp.linalg.norm(W[k][:, emission_dim:])
        print(f"State {k}: ||W_lag1|| = {lag1_norm:.4f}, ||W_lag2|| = {lag2_norm:.4f}")

---
## 10. Detailed Look: What `compute_inputs` Produces for Multiple Lags

Let's trace through exactly what the input matrix looks like.

In [ ]:
# Small example to see the structure clearly
small_emissions = jnp.array([
    [1.0, 2.0],   # t=0
    [3.0, 4.0],   # t=1
    [5.0, 6.0],   # t=2
    [7.0, 8.0],   # t=3
    [9.0, 10.0],  # t=4
])

for n_lags in [1, 2, 3]:
    model_tmp = LinearAutoregressiveHMM(2, 2, num_lags=n_lags)
    inp = model_tmp.compute_inputs(small_emissions)
    print(f"\nnum_lags = {n_lags}, input shape: {inp.shape}")
    print(f"  Columns are: [{', '.join([f'y(t-{l})' for l in range(1, n_lags+1)])}] (each 2D)")
    print(f"  Row t=0: {inp[0]}  (padded with zeros for missing history)")
    print(f"  Row t=1: {inp[1]}")
    print(f"  Row t=2: {inp[2]}")
    if n_lags <= 2:
        print(f"  Row t=3: {inp[3]}")
        print(f"  Row t=4: {inp[4]}")

---
## 11. Summary: Class Hierarchy

Understanding the code architecture helps when you want to extend it.

```
HMM (abstract base)                        # dynamax/hmm/models/abstractions.py
 ├── initial_component: StandardHMMInitialState
 ├── transition_component: StandardHMMTransitions
 └── emission_component: HMMEmissions

LinearRegressionHMM(HMM)                   # linreg_hmm.py
 └── emission_component: LinearRegressionHMMEmissions
     - distribution(state, inputs):  N(W[state] @ inputs + b[state], Σ[state])
     - collect_suff_stats():  weighted outer products for M-step
     - m_step():              solve weighted least-squares

LinearAutoregressiveHMM(HMM)               # arhmm.py
 └── emission_component: LinearAutoregressiveHMMEmissions(LinearRegressionHMMEmissions)
     - Overrides: initialize() — sets lag-1 weights to near-identity (0.95*I)
     - Inherits:  distribution(), collect_suff_stats(), m_step() — unchanged!
     + New:       compute_inputs() — creates lagged input matrix
     + New:       sample() — maintains prev_emissions buffer
```

### Key takeaway
The AR-HMM **reuses** the linear regression HMM's EM code entirely.
The only AR-specific code is:
1. `compute_inputs()` — builds the lagged feature matrix
2. `sample()` — maintains a sliding window of past emissions
3. `initialize()` — sets sensible priors (near-identity for lag-1, near-zero for others)

---
## 12. Next Steps: Connecting to SLDS for ECoG

Now that you understand how lags work in AR-HMMs, here's how this connects
to your ECoG work with SLDS:

### AR-HMM vs. SLDS with lags

| | AR-HMM (this notebook) | SLDS (your ECoG model) |
|---|---|---|
| **What has lags** | Observations directly | Latent continuous state |
| **Model** | $y_t = A_k y_{t-1} + b_k + \epsilon$ | $x_t = A_k x_{t-1} + b_k + w_t$, $y_t = C x_t + d + v_t$ |
| **With multi-lag** | $y_t = \sum_\ell W_{k,\ell} y_{t-\ell} + b_k + \epsilon$ | $x_t = \sum_\ell A_{k}^{(\ell)} x_{t-\ell} + b_k + w_t$ |
| **Inference** | Exact (forward-backward) | Approximate (variational) |

### In the ssm library
Your existing SLDS code can add lags with one line:
```python
slds = ssm.SLDS(N, K, D,
                dynamics="gaussian",
                dynamics_kwargs=dict(lags=2))  # <-- this is all you need
```

### Practical considerations for ECoG
1. **Sampling rate**: At 1000 Hz, lag-1 = 1ms. You may need lag-5 to lag-20
   depending on the frequency band of interest, or consider downsampling first.
2. **Model selection**: Use AIC/BIC or cross-validation to pick lag order
3. **Inference**: With `lags > 1` in ssm, use BBVI (not Laplace-EM)
4. **Start simple**: Try AR-HMM on your ECoG data first (no latent continuous
   state), then add the SLDS layer once you understand the lag structure